In [ ]:
import fenic as fc

from dotenv import load_dotenv

fc.configure_logging()

load_dotenv()

config = fc.SessionConfig(
        app_name="medium_curation",
        semantic=fc.SemanticConfig(
            language_models={
                "flash": fc.GoogleGLAModelConfig(
                    model_name="gemini-2.0-flash",
                    rpm=2000,
                    tpm=4_000_000,
                ),
            },
            embedding_models={
                "large": fc.OpenAIModelConfig(
                    model_name="text-embedding-3-large",
                    rpm=3000,
                    tpm=1_000_000
                )
            }
        ),
    )

session = fc.Session.get_or_create(config)

Script 2: Is On Topic

In [ ]:
with_on_topic_label = session.table("with_on_topic_label")
with_on_topic_label.group_by("is_on_topic").agg(fc.count("*")).show()
with_on_topic_label.filter(~fc.col("is_on_topic")).select("title", "text").show(5)

Script 3: Extract Features

In [ ]:
with_features = (
    session.table("with_features")
    .select(
        "title",
        "topic_annotation",
        "has_code",
        "technical_terms",
        "form_factor",
        "technical_complexity",
    )
)
with_features.show(10)


Script 4: Technical Terms

In [ ]:
with_technical_terms = session.table("with_cleaned_technical_terms")

with_technical_terms.select("cleaned_terms") \
    .explode("cleaned_terms") \
    .group_by("cleaned_terms") \
    .agg(fc.count("*").alias("count")) \
    .order_by(fc.col("count").desc()) \
    .show(30)


Script 5: Technical Complexity 

In [ ]:
with_complexity_label = session.table("with_complexity_label")

with_complexity_label.select("complexity_label") \
    .group_by("complexity_label") \
    .agg(fc.count("*").alias("count")) \
    .order_by(fc.col("count").desc()) \
    .show()

with_complexity_label.select("title", "text", "complexity_label").filter(fc.col("complexity_label") == "low").show(5)
with_complexity_label.select("title", "text", "complexity_label").filter(fc.col("complexity_label") == "high").show(4)

Script 6: Narrative Intent

In [ ]:
with_narrative_intent = session.table("with_narrative_intent_label")

with_narrative_intent.select("narrative_intent_label") \
    .group_by("narrative_intent_label") \
    .agg(fc.count("*").alias("count")) \
    .order_by(fc.col("count").desc()) \
    .show()

with_narrative_intent.select("title", "text", "narrative_intent_label").filter(fc.col("narrative_intent_label") == "inform").show(5)
with_narrative_intent.select("title", "text", "narrative_intent_label").filter(fc.col("narrative_intent_label") == "teach").show(5)
with_narrative_intent.select("title", "text", "narrative_intent_label").filter(fc.col("narrative_intent_label") == "persuade").show(5)

Script 7: Topic Modeling

In [ ]:
with_topic_labels = session.table("with_topic_labels")

with_topic_labels.select("topic_name") \
    .group_by("topic_name") \
    .agg(fc.count("*").alias("count")) \
    .order_by(fc.col("count").desc()) \
    .show()

with_subtopic_labels = session.table("with_subtopic_labels")

with_subtopic_labels.select("sub_topic_name") \
    .group_by("sub_topic_name") \
    .agg(fc.count("*").alias("count")) \
    .order_by(fc.col("count").desc()) \
    .show()

Script 8: Finalize

In [ ]:
with_final_labels = session.table("with_final_labels")
with_final_labels.show(5)
print(with_final_labels.schema)

In [41]:
session.stop()